<a href="https://colab.research.google.com/github/stiltnerag/INFO648/blob/main/Homework_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Part 1: Class Distribution and a Baseline

In [1]:
import pandas as pd

In [3]:
df=pd.read_csv('/content/churn_synthetic_600.csv')

In [4]:
df.head()

,CustomerID,Churn,TenureMonths,MonthlyCharge,ContractType,PaymentMethod,SupportCalls
0,C0001,1,20,86.27,One-Year,Credit Card,3
1,C0002,0,1,64.80,Month-to-Month,Electronic Check,1
2,C0003,1,20,75.02,Month-to-Month,Credit Card,3
3,C0004,0,18,61.21,Month-to-Month,Bank Transfer,2
4,C0005,0,63,66.20,Two-Year,Bank Transfer,0


In [52]:
df.describe()

,Churn,TenureMonths,MonthlyCharge,SupportCalls
count,600.000000,600.000000,600.000000,600.00000
mean,0.236667,37.461667,69.517050,1.22000
std,0.425391,20.299967,19.687343,1.18347
min,0.000000,1.000000,25.000000,0.00000
25%,0.000000,20.000000,56.697500,0.00000
50%,0.000000,38.000000,68.765000,1.00000
75%,0.000000,54.000000,82.845000,2.00000
max,1.000000,72.000000,138.990000,6.00000


In [54]:
TenureMonths=df['TenureMonths'].median()

In [55]:
TenureMonths

38.0

In [58]:
model_df['Churn']=(model_df['TenureMonths']>38).astype(int)

In [59]:
model_df=df.copy()

In [60]:
churn_counts = df['Churn'].value_counts()
churn_proportions = df['Churn'].value_counts(normalize=True)

In [61]:
churn_report = pd.DataFrame({
    'Count': churn_counts,
    'Proportion': churn_proportions,
    'Percentage (%)': churn_proportions * 100
})

print(churn_report)

       Count  Proportion  Percentage (%)
Churn                                   
0        458    0.763333       76.333333
1        142    0.236667       23.666667


In [62]:
class_proportions = df['Churn'].value_counts(normalize=True)

In [14]:
trivial_model_accuracy = class_proportions.max()

In [15]:
print(f"Trivial Model Accuracy (Majority Class Baseline): {trivial_model_accuracy:.4f}")
print(f"As a percentage: {trivial_model_accuracy * 100:.2f}%")

Trivial Model Accuracy (Majority Class Baseline): 0.7633
As a percentage: 76.33%


1. The dataset is imbalanced, 3:1 for everyone 1 customer who churns we have more than 3 customers who stay.

2. Class imbalance complicates evaluation because metrics like accuracy treat all predictions equally, regardless of class. In our dataset 76% of the data belongs to class 0 (no churn), and the model doesn't actually learn how to identify a churned customer as is.

3. It could be misleading because it is failing to catch the 142 customers who are actually leaving, we need to detect the churn not just boast a high retention percentage.

#Part 2: Preprocessing Plan

Numeric:

*  TenureMonths
*  MonthlyCharge
*  SupportCalls

Categorical:

*   ContractType
*   PaymentMethod
*   Churn




In [41]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [22]:
numeric_features = ['TenureMonths', 'MonthlyCharge', 'SupportCalls']
categorical_features = ['ContractType', 'PaymentMethod']

It is necessary because of the way the algorithm performs arithmetic operations on these inputs, it cannot natively process text strings like "Month-to-Month" or "Credit Card". So, one-hot encoding converts these qualitative categories into binary numbers, which allows the model to compute a distinct numeric weight for the presence or absence of each category.

Fitting your preprocessing transformations inside a machine learning pipeline—rather than applying them to the entire dataset beforehand—is critical in preventing data leakage.

The pipeline guarantees that the scaler will only fit on the training portion of the data during cross-validation. This allows for real world scenarios to be mimicked to achieve honest and unbiased estimates for the model's performance.

#Part 3: Build the Pipeline

In [35]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

X = df.drop(columns=['CustomerID', 'Churn'])
y = df['Churn']

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [64]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [65]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, solver='liblinear'))
])

In [66]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['TenureMonths',
                                                   'MonthlyCharge',
                                                   'SupportCalls']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['ContractType',
                                                   'PaymentMethod'])])),
                ('classifier',
                 LogisticRegression(random_state=42, solver='liblinear'))])

In [45]:
numeric_features = ['TenureMonths', 'MonthlyCharge', 'SupportCalls']
categorical_features = ['ContractType', 'PaymentMethod']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ],
    remainder='drop' # Automatically drops CustomerID if it wasn't already removed
)

In [46]:
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(random_state=42))
    ]
)

In [47]:
pipeline.fit(X_train, y_train)

print("Pipeline successfully fitted on training data!")

Pipeline successfully fitted on training data!


We used a stratified split because the analysis in Part 1 showed our dataset was imbalanced, we want to rule out any sampling bias. Using a stratified split we are forcing the ratio in both datasets (training set and test set) to remain exact.

#Part 4: Evaluation

In [67]:
y_pred = pipeline.predict(X_test)
y_pred

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [71]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")



Accuracy: 0.8000
Precision: 0.6000
Recall: 0.4286
F1-Score: 0.5000


In [69]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[84  8]
 [16 12]]


In [72]:
from sklearn.metrics import confusion_matrix, classification_report
#Actual first
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred, digits=3))


[[84  8]
 [16 12]]
              precision    recall  f1-score   support

           0      0.840     0.913     0.875        92
           1      0.600     0.429     0.500        28

    accuracy                          0.800       120
   macro avg      0.720     0.671     0.688       120
weighted avg      0.784     0.800     0.787       120



The four cells of the confusion matrix represents true negatives 84 loyal customers who did not churn and the model correctly predicited they would stay.

False positives - 8 loyal customers who the model mistakenly flagged as risk of churning, but they truly weren't planning to leave.

False negatives - 16 customers who actually churned, but the model missed them and predicited they would stay.

True Positives - 12 customers who churned and the model caught them. The people we want to target to actually retain.

Precision is the 12/ (12+8), so the 0.600, only 60% of the total predicted churns acutally happened, so we had a 40% false alarm.

Recall is 12/(12+16), so .429, 42.9% meaning we missed over half of the churning population.

The new model accuracy is 80% which is slightly higher than the inital 76.33% we saw. Our model also identified 12 real churners, but it did miss more churners than it found.

#Part 5: Interpreting the Coefficients

In [76]:
fitted_preprocessor = pipeline.named_steps['preprocessor']
fitted_model = pipeline.named_steps['classifier']

In [77]:
feature_names = fitted_preprocessor.get_feature_names_out()

In [78]:
coefficients = fitted_model.coef_.flatten()

In [79]:
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', ascending=False)

print("--- All Features Ranked by Coefficient ---")
print(coef_df.to_string(index=False))

--- All Features Ranked by Coefficient ---
                            Feature  Coefficient
                 num__MonthlyCharge     0.584985
                  num__SupportCalls     0.397859
   cat__ContractType_Month-to-Month     0.395997
cat__PaymentMethod_Electronic Check    -0.238948
     cat__PaymentMethod_Credit Card    -0.402760
   cat__PaymentMethod_Bank Transfer    -0.424085
         cat__ContractType_One-Year    -0.441247
                  num__TenureMonths    -0.913717
         cat__ContractType_Two-Year    -1.020543


Looking at the num_support calls with a positive coefficient of 0.398, we are able to tell that as the number of customer support calls increase, the customer becomes more likely to churn. When we compare this to other coefficients we see that it roughly matches customers being volatile month-to-month at 0.396. This tells us from a business perspective when a customer logs a certain number of calls (3/4) then we want to put them into a category of high risk of churn.

#Part 6: The Precision–Recall Tradeoff

Recalling part 4 our precision was 60% and recall was 42.9%, the precision means that our model is producing false alarms at a rather high rate. If we are handing out incentives to stay we are eating into revenue on customers who had no intention of leaving. The low recall of 42.9% is showing our blind spot, the model is missing 57% customers who leave without us ever trying to incentivize to stay. The cost of losing a customer would outweigh the cost of trying to get them to stay, however we want to be targeting the right group. Therefore, I would prioritize recall, realizing that doing so will allow in a few more false alarms which will lower the precision this is an exchange I'm willing to make if it will allow us to catch a larger percentage of the customers slipping through the cracks. The loss of revenue on churn customers would be outweighed, especially if we implemented a low-cost retention offer.  